## परिचय

हा धडा यावर लक्ष केंद्रित करेल:
- फंक्शन कॉलिंग म्हणजे काय आणि त्याच्या वापराच्या बाबतीत
- Azure OpenAI वापरून फंक्शन कॉल कसे तयार करायचे
- फंक्शन कॉलला अॅप्लिकेशनमध्ये कसे एकत्रित करायचे

## शिक्षणाचे उद्दिष्टे

हा धडा पूर्ण केल्यावर तुम्हाला काय करायचे आहे आणि समजेल:

- फंक्शन कॉलिंगचा उद्देश काय आहे
- Azure Open AI सर्व्हिस वापरून फंक्शन कॉल सेटअप करणे
- तुमच्या अॅप्लिकेशनच्या वापरासाठी प्रभावी फंक्शन कॉल डिझाइन करणे


## फंक्शन कॉल्स समजून घेणे

या धड्यात, आम्हाला आमच्या शिक्षण स्टार्टअपने एक अशी सुविधा तयार करायची आहे जिथे वापरकर्ते तांत्रिक अभ्यासक्रम शोधण्यासाठी चॅटबॉटचा वापर करू शकतील. आम्ही त्यांच्या कौशल्याच्या पातळी, सध्याच्या भूमिके आणि आवडत्या तंत्रज्ञानानुसार अभ्यासक्रमांची शिफारस करू.

हे पूर्ण करण्यासाठी आम्ही खालील संयोजन वापरणार आहोत:
 - वापरकर्त्यांसाठी चॅट अनुभव निर्माण करण्यासाठी `Azure Open AI`
 - वापरकर्त्यांच्या विनंतीनुसार अभ्यासक्रम शोधण्यात मदत करण्यासाठी `Microsoft Learn Catalog API`
 - वापरकर्त्यांच्या क्वेरीला फंक्शनकडे पाठवण्यासाठी आणि API विनंती करण्यासाठी `Function Calling`

प्रारंभ करण्यासाठी, आपण पाहू की आम्हाला फंक्शन कॉलिंग का वापरायचे आहे:


### फंक्शन कॉलिंग का आहे

जर तुम्ही या कोर्समधील इतर कोणतेही धडा पूर्ण केले असेल, तर तुम्हाला Large Language Models (LLMs) वापरण्याची ताकद कदाचित समजली असेल. आशा आहे की तुम्हाला त्यांची काही मर्यादा देखील समजल्या असतील.

फंक्शन कॉलिंग हे Azure Open AI Service चे एक वैशिष्ट्य आहे जे खालील मर्यादांवर मात करते:
1) सुसंगत प्रतिसाद फॉर्मॅट
2) चॅट संदर्भात अनुप्रयोगाच्या इतर स्रोतांमधून डेटा वापरण्याची क्षमता

फंक्शन कॉलिंगपूर्वी, LLM कडून मिळालेले प्रतिसाद अनस्ट्रक्चर्ड आणि असुसंगत होते. विकसित करणाऱ्यांना प्रत्येक प्रतिसादाच्या विविधतेला हाताळण्यासाठी क्लिष्ट व्हॅलिडेशन कोड लिहायचा अनिवार्य होता.

वापरकर्ते "स्टॉकहोममधील सध्याचे हवामान काय आहे?" यांसारखे प्रश्न विचारू शकले नाहीत. कारण मॉडेल्स केवळ डेटा प्रशिक्षणाच्या वेळेपर्यंत मर्यादित होते.

खालील उदाहरण पाहू ज्यामुळे ही समस्या स्पष्ट होते:

समजा आपण विद्यार्थ्यांचा डेटा ठेवण्यासाठी एक डेटाबेस तयार करू इच्छितो जेणेकरून त्यांना योग्य कोर्स सुचवता येईल. खाली विद्यार्थ्यांचे दोन वर्णन आहेत जे त्यांच्या डेटामध्ये खूपच सारखे आहेत.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


आम्हाला हे LLM कडे पाठवायचे आहे जेणेकरून ते डेटा पार्स करू शकेल. नंतर हे आमच्या अनुप्रयोगात API कडे पाठवण्यासाठी किंवा डेटाबेसमध्ये साठवण्यासाठी वापरले जाऊ शकते.

चला दोन एकसारखे प्रॉम्प्ट तयार करूया ज्याद्वारे आपण LLM ला कळवू की आपण कोणती माहिती जाणून घेऊ इच्छित आहोत:


आम्हाला हे आमच्या उत्पादनासाठी महत्त्वाच्या भागांचे विश्लेषण करण्यासाठी LLM कडे पाठवायचे आहे. म्हणून आपण LLM ला निर्देश देण्यासाठी दोन एकसारखे प्रॉम्प्ट तयार करू शकतो: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


या दोन प्रॉम्प्ट्स तयार केल्यानंतर, आपण त्यांना `client.responses.create` वापरून LLM कडे पाठवू. आपण प्रॉम्प्ट `input` व्हेरिएबलमध्ये साठवतो आणि भूमिका `user` ला देते. हे वापरकर्त्याने चॅटबॉटसाठी संदेश लिहिल्याचा अनुकरण करण्यासाठी आहे.



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

आता आपण दोन्ही विनंत्या LLM कडे पाठवू आणि आम्हाला मिळालेला प्रतिसाद तपासू शकतो. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



जरी प्रविष्ट्या एकसारख्या असल्या आणि वर्णने सुमारेसुमारे असली तरी, आपल्याला `Grades` गुणधर्माच्या वेगवेगळ्या स्वरूपात प्राप्त होऊ शकतात.

जर आपण वरील सेल अनेकदा चालवले, तर स्वरूप `3.7` किंवा `3.7 GPA` असू शकते.

कारण LLM लिहिलेल्या प्रविष्ट्याच्या रूपात असंरचित डेटा घेतो आणि परत देखील असंरचित डेटा परत करतो. आपल्याला संरचित स्वरूप असावे लागते जेणेकरून जेव्हा आपण डेटा संग्रहित करता किंवा वापरता तेव्हा काय अपेक्षा करायची ते कळेल.

फंक्शन कॉलिंग वापरून, आपण हे सुनिश्चित करू शकतो की आपल्याला संरचित डेटा परत मिळेल. फंक्शन कॉलिंग वापरताना, LLM प्रत्यक्षात कोणतेही फंक्शन कॉल किंवा चालवत नाही. त्याऐवजी, आपण LLM ला त्याच्या प्रतिसादांसाठी पालन करण्यासाठी एक संरचना तयार करतो. नंतर आपण त्या संरचित प्रतिसादांचा वापर करून आपल्या अनुप्रयोगांमध्ये कोणते फंक्शन चालवायचे ते ठरवतो.
 


![फंक्शन कॉलिंग प्रवाह आरेख](../../../../translated_images/mr/Function-Flow.083875364af4f4bb.webp)


नंतर आपण फंक्शनमधून परत येणारे मूल्य घेऊन ते परत LLM कडे पाठवू शकतो. नंतर LLM नैसर्गिक भाषेत प्रतिसाद देऊन वापरकर्त्याच्या प्रश्नाचे उत्तर देईल.


### फंक्शन कॉल वापरण्याचे वापर प्रकरणे 

**बाह्य साधनांना कॉल करणे** 
चॅटबॉट्स वापरकर्त्यांच्या प्रश्नांसाठी उत्तरे देण्यात उत्कृष्ट असतात. फंक्शन कॉलिंग वापरून, चॅटबॉट्स वापरकर्त्यांकडून आलेल्या संदेशांचा उपयोग करून विशिष्ट कार्ये पूर्ण करू शकतात. उदाहरणार्थ, विद्यार्थी चॅटबॉटला विचारू शकतो की "माझ्या शिक्षकांना ईमेल पाठवा की मला या विषयाबाबत अधिक मदत हवी आहे". हे `send_email(to: string, body: string)` या फंक्शनला कॉल करू शकते.


**API किंवा डेटाबेस क्वेरी तयार करणे**
वापरकर्ते नैसर्गिक भाषेत माहिती शोधू शकतात ज्याला नंतर फॉर्मॅटेड क्वेरी किंवा API विनंतीमध्ये रूपांतरित केले जाते. याचे उदाहरण म्हणजे शिक्षक जो विचारतो "कोणते विद्यार्थी शेवटचे असाइनमेंट पूर्ण केले आहेत" ज्यामुळे `get_completed(student_name: string, assignment: int, current_status: string)` नावाच्या फंक्शनला कॉल होऊ शकते.


**रचनेत माहिती तयार करणे**
वापरकर्ते मजकूराचा ब्लॉक किंवा CSV घेऊन LLM चा वापर करून त्यातील महत्त्वाची माहिती काढू शकतात. उदाहरणार्थ, विद्यार्थी शांतता करारांबद्दल विकिपीडिया लेख रूपांतरित करून AI फ्लॅश कार्ड तयार करू शकतो. हे `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)` नावाचा फंक्शन वापरून करता येईल.


## 2. तुमच्या पहिल्या फंक्शन कॉलची निर्मिती 

फंक्शन कॉल तयार करण्याची प्रक्रिया ३ मुख्य टप्प्यांत विभागलेली आहे: 
1. तुमच्या फंक्शन्सची यादी आणि वापरकर्त्याचा संदेश देऊन Chat Completions API ला कॉल करणे 
2. क्रिया पार पाडण्यासाठी मॉडेलच्या प्रतिसादाचा अभ्यास करणे म्हणजे फंक्शन किंवा API कॉल चालवणे 
3. फंक्शनमधून प्राप्त प्रतिसाद वापरून वापरकर्त्यासाठी प्रतिसाद तयार करण्यासाठी Chat Completions API ला आणखी एक कॉल करणे. 


![फंक्शन कॉलचा प्रवाह](../../../../translated_images/mr/LLM-Flow.3285ed8caf4796d7.webp)


### फंक्शन कॉलची घटकं 

#### वापरकर्त्याची इनपुट 

पहिला टप्पा म्हणजे वापरकर्त्याचा संदेश तयार करणे. हे डायनॅमिकली टेक्स्ट इनपुटमधील मूल्य घेऊन नियुक्त केले जाऊ शकते किंवा तुम्ही येथे एक मूल्य नियुक्त करू शकता. जर तुम्ही Chat Completions API सह प्रथमच काम करत असाल, तर तुम्हाला संदेशाचा `role` आणि `content` निश्चित करावा लागेल. 

`role` हा `system` (नियम तयार करणे), `assistant` (मॉडेल) किंवा `user` (अंतिम वापरकर्ता) पैकी काहीही असू शकतो. फंक्शन कॉलसाठी, आपण याला `user` म्हणून आणि एक उदाहरण प्रश्न म्हणून नियुक्त करू. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### फंक्शन्स तयार करणे.

पुढे आपण एक फंक्शन आणि त्या फंक्शनचे पॅरामिटर्स परिभाषित करू. येथे आपण `search_courses` नावाचे एकच फंक्शन वापरणार आहोत पण तुम्ही एकाहून अधिक फंक्शन्स तयार करू शकता.

**महत्वाचे** : फंक्शन्स LLM ला दिलेल्या सिस्टम मेसेजमध्ये समाविष्ट असतात आणि तुमच्याकडे उपलब्ध असलेल्या टोकनच्या क्षमतेत त्यांचा समावेश होतो.


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**व्याख्या** 

`name` - कॉल करण्यासाठी आम्हाला हवी असलेली फंक्शनचे नाव. 

`description` - फंक्शन कसे कार्य करते याचे वर्णन. येथे विशिष्ट आणि स्पष्ट असणे महत्त्वाचे आहे 

`parameters` - मॉडेलला त्याच्या प्रतिसादात तयार करण्यासाठी हवा असलेला मूल्य आणि स्वरूपांची यादी 


`type` - मालमत्ता ज्या डेटा प्रकारात साठवली जातील 

`properties` - विशिष्ट मूल्यांची यादी जी मॉडेल आपल्या प्रतिसादासाठी वापरेल 


`name` - मॉडेल आपल्या स्वरूपित प्रतिसादात वापरणार्‍या मालमत्तेचे नाव 

`type` - या मालमत्तेचा डेटा प्रकार 

`description` - विशिष्ट मालमत्तेचे वर्णन 


**ऐच्छिक**

`required` - फंक्शन कॉल पूर्ण करण्यासाठी आवश्यक मालमत्ता 


### फंक्शन कॉल करणे  
फंक्शन परिभाषित केल्यानंतर, आता आपल्याला ते Chat Completion API च्या कॉलमध्ये समाविष्ट करणे आवश्यक आहे. आपण हे विनंतीत `functions` जोडून करतो. या प्रकरणात `functions=functions`.  

`function_call` ला `auto` सेट करण्याचा एक पर्याय देखील आहे. याचा अर्थ असा की आपण स्वतः फंक्शन नियुक्त न करता वापरकर्त्याच्या संदेशावर आधारित कोणते फंक्शन कॉल करायचे याचा निर्णय LLM ला द्याल.  


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


चला आता प्रतिसादाकडे पाहूया आणि ते कसे स्वरूपित केले आहे ते पाहूया: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

तुम्हाला दिसेल की फंक्शनचे नाव कॉल केले आहे आणि वापरकर्त्याच्या संदेशावरून, LLM ला फंक्शनच्या arguments साठी डेटा शोधता आला आहे. 


## 3.अ‍ॅप्लिकेशनमध्ये फंक्शन कॉल्स एकत्रित करणे.


LLM कडून फॉरमॅट केलेल्या प्रतिसादाची चाचणी घेतल्यानंतर, आता आपण याला एका अ‍ॅप्लिकेशनमध्ये एकत्रित करू शकतो.

### प्रवाह व्यवस्थापन

याला आपल्या अ‍ॅप्लिकेशनमध्ये एकत्रित करण्यासाठी, चला पुढील पावले घेऊया:

प्रथम, Open AI सेवांशी कॉल करूया आणि संदेश `response_message` नावाच्या व्हेरिएबलमध्ये संग्रहित करूया.


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


आता आपण कार्य परिभाषित करू जे Microsoft Learn API ला कॉल करेल जेणेकरून कोर्सेसची यादी मिळवता येईल: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



एक सर्वोत्तम पद्धत म्हणून, आपण पाहणार आहोत की मॉडेलला एखादी फंक्शन कॉल करायची आहे का. त्यानंतर, उपलब्ध फंक्शन्सपैकी एक तयार करू आणि त्याला कॉल होणाऱ्या फंक्शनशी जुळवू. 
त्यानंतर आपण फंक्शनचे आर्ग्युमेंट्स घेऊन त्यांना LLM मधील आर्ग्युमेंट्सशी मॅप करू.

शेवटी, आपण फंक्शन कॉल मेसेज आणि `search_courses` मेसेजने परत दिलेल्या मूल्यांना जोडू. यामुळे LLM कडे वापरकर्त्याला नैसर्गिक भाषेत प्रतिसाद देण्यासाठी सर्व माहिती मिळते. 
प्रतिसाद देण्यासाठी आवश्यक सर्व माहिती मिळते. 


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


आता आपण अद्ययावत संदेश LLM ला पाठवू ज्यामुळे आपण API JSON स्वरूपातील उत्तराऐवजी नैसर्गिक भाषेचा प्रतिसाद प्राप्त करू शकू. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## कोड आव्हान 

उत्कृष्ट कामगिरी! Azure Open AI फंक्शन कॉलिंगचा तुमचा अभ्यास सुरू ठेवण्यासाठी तुम्ही तयार करू शकता: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - फंक्शनच्या अधिक पॅरामीटर्स जे शिकणाऱ्यांना अधिक अभ्यासक्रम शोधण्यात मदत करू शकतात. उपलब्ध API पॅरामीटर्स येथे सापडतील: 
 - आणखी एक फंक्शन कॉल तयार करा जो शिकणाऱ्यांकडून त्यांच्या मातृभाषेसारखी अधिक माहिती घेईल 
 - जेव्हा फंक्शन कॉल आणि/किंवा API कॉल योग्य अभ्यासक्रम परत करत नाही तेव्हा त्रुटी हाताळणी तयार करा 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
